# Stage 3: Data Cleaning & Transformation

**Financial Forecasting & Variance Analysis** | Stage 3 of 6 | ← [Stage 2: Data Ingestion](../02-data-ingestion/02-data-ingestion.ipynb) | [Project README](../../README.md) | [Stage 4: Data Modelling & Analytics](../04-data-modelling-and-analytics/04-data-modelling-and-analytics.ipynb) →

> Turn the text-only bronze tables into typed, consistently named silver tables, then prove their integrity with SQL data-quality controls.

| Reads | Writes |
| --- | --- |
| `data/bronze/*.parquet` | `data/silver/*.parquet`, `data/gold/data_quality_checks.parquet` |

## 3.1 Cleaning rules

| # | Table | Rule | Why |
| --- | --- | --- | --- |
| C1 | all | Rename keys to one convention: `id` becomes `user_id` / `card_id` / `transaction_id`, `client_id` becomes `user_id`, `mcc` becomes `mcc_code` | Joins read naturally and can't silently match the wrong `id` |
| C2 | transactions, users, cards | Parse currency text to numbers: `$1,234`, `$-77.00` and `($77.00)` all become signed floats | The source mixes these formats; values that can't be parsed become null and are caught by control NP01/NP04 |
| C3 | transactions | Keep negative amounts as **refunds/reversals** and flag them with `is_refund` | Net spend = purchases − refunds. Dropping refunds would overstate spend |
| C4 | cards | Parse `expires` **and** `acct_open_date` as `MM/YYYY` | The original notebook parsed `acct_open_date` as `MM/DD/YYYY`, which turned every value into NaT |
| C5 | cards | Drop `card_number` and `cvv` | No analytical use; card-like data shouldn't travel downstream |
| C6 | transactions | Leave missing `merchant_state`/`zip` on online transactions as **null** and add `is_online` | Filling them with `'unknown'` hides the fact that the location doesn't exist |
| C7 | transactions | Treat a missing `errors` value as no error and add `has_error` | An empty error field means the transaction went through cleanly |
| C8 | fraud labels | Map `Yes`/`No` to a boolean `is_fraud` | The labels are reshaped from a JSON dictionary in bronze |
| C9 | users, cards | Map `YES`/`NO` flags to booleans | Consistent types for aggregation |

The code is in [`python/cleaning.py`](../../python/cleaning.py). Each function returns a new DataFrame and never changes the bronze table it reads.

In [1]:
import sys
from pathlib import Path

# Locate the repo root (the folder that holds python/config.py) and import the shared modules
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "python" / "config.py").exists())
sys.path.insert(0, str(ROOT / "python"))

import pandas as pd
import config

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [2]:
from cleaning import build_silver, read_silver
from ingestion import read_bronze

silver_log = build_silver()
silver_log

,table,bronze_rows,silver_rows,silver_columns
0,users,2000,2000,14
1,cards,6146,6146,11
2,mcc_codes,109,109,2
3,fraud_labels,8914963,8914963,2
4,transactions,13305915,13305915,15


## 3.2 Before and after: rules C2 and C4

A few rows side by side, to show the parsing behaves as described.

In [3]:
bronze_tx = read_bronze("transactions")[["id", "amount", "date"]].head(1_000)
silver_tx = read_silver("transactions")[["transaction_id", "amount", "is_refund", "transaction_date"]].head(1_000)
pd.concat([bronze_tx.add_prefix("bronze_"), silver_tx.add_prefix("silver_")], axis=1).loc[
    lambda d: d["silver_is_refund"] | (d.index < 3)
].head(6)

,bronze_id,bronze_amount,bronze_date,silver_transaction_id,silver_amount,silver_is_refund,silver_transaction_date
0,7475327,$-77.00,2010-01-01 00:01:00,7475327,-77.00,True,2010-01-01 00:01:00
1,7475328,$14.57,2010-01-01 00:02:00,7475328,14.57,False,2010-01-01 00:02:00
2,7475329,$80.00,2010-01-01 00:02:00,7475329,80.00,False,2010-01-01 00:02:00
19,7475347,$-64.00,2010-01-01 00:36:00,7475347,-64.00,True,2010-01-01 00:36:00
50,7475382,$-78.00,2010-01-01 01:25:00,7475382,-78.00,True,2010-01-01 01:25:00
81,7475422,$-460.00,2010-01-01 02:13:00,7475422,-460.00,True,2010-01-01 02:13:00


In [4]:
bronze_cards = read_bronze("cards")[["id", "expires", "acct_open_date", "credit_limit"]].head(3)
silver_cards = read_silver("cards")[["card_id", "expires", "acct_open_date", "credit_limit"]].head(3)
pd.concat([bronze_cards.add_prefix("bronze_"), silver_cards.add_prefix("silver_")], axis=1)

,bronze_id,bronze_expires,bronze_acct_open_date,bronze_credit_limit,silver_card_id,silver_expires,silver_acct_open_date,silver_credit_limit
0,4524,12/2022,09/2002,$24295,4524,2022-12-01,2002-09-01,"24,295.00"
1,2731,12/2020,04/2014,$21968,2731,2020-12-01,2014-04-01,"21,968.00"
2,3701,02/2024,07/2003,$46414,3701,2024-02-01,2003-07-01,"46,414.00"


In [5]:
del bronze_tx, silver_tx, bronze_cards, silver_cards

## 3.3 Data-quality controls

[`sql/quality/data_quality_checks.sql`](../../sql/quality/data_quality_checks.sql) runs in DuckDB directly against the silver Parquet files.

- **Controls** (`PK`, `RI`, `NP`) should flag **zero** rows: primary-key uniqueness, referential integrity, and values that failed to parse.
- **Profiles** (`PR`) describe the data: refunds, errors, online share and fraud-label coverage. They don't pass or fail, but later stages depend on them.

In [6]:
import warehouse

con = warehouse.connect()
dq = warehouse.run_sql_file(con, "quality/data_quality_checks.sql")
dq

,check_id,severity,table_name,check_name,flagged_rows,total_rows,flagged_pct,status
0,NP01,control,transactions,amount failed to parse,0,13305915,0.00,PASS
1,NP02,control,transactions,transaction_date failed to parse,0,13305915,0.00,PASS
2,NP03,control,cards,acct_open_date or expires failed to parse,0,6146,0.00,PASS
3,NP04,control,users,income or debt failed to parse,0,2000,0.00,PASS
4,NP05,control,fraud_labels,label not Yes/No,0,8914963,0.00,PASS
5,PK01,control,users,duplicate user_id,0,2000,0.00,PASS
6,PK02,control,cards,duplicate card_id,0,6146,0.00,PASS
7,PK03,control,mcc_codes,duplicate mcc_code,0,109,0.00,PASS
8,PK04,control,transactions,duplicate transaction_id,0,13305915,0.00,PASS
9,PK05,control,fraud_labels,duplicate transaction_id,0,8914963,0.00,PASS


In [7]:
summary = dq.groupby(["severity", "status"]).size().rename("checks").reset_index()
display(summary)

failed = dq.loc[dq["status"] == "FAIL"]
if failed.empty:
    print("All integrity controls pass.")
else:
    print("Controls that flagged rows. Each must be explained or fixed before stage 4:")
    display(failed[["check_id", "table_name", "check_name", "flagged_rows", "flagged_pct"]])

warehouse.write_gold(con, "data_quality_checks", dq)

,severity,status,checks
0,control,PASS,16
1,profile,INFO,6


All integrity controls pass.


## 3.4 Coverage profile

The date range, month completeness and yearly volumes set the limits for stage 4: how many months the forecast can learn from, and which edge months are excluded.

In [8]:
coverage = warehouse.query(con, """
    SELECT
        MIN(transaction_date)                                     AS first_transaction,
        MAX(transaction_date)                                     AS last_transaction,
        COUNT(DISTINCT date_trunc('month', transaction_date))     AS calendar_months,
        COUNT(DISTINCT user_id)                                   AS customers_transacting,
        COUNT(DISTINCT card_id)                                   AS cards_used,
        COUNT(DISTINCT merchant_id)                               AS merchants
    FROM silver_transactions
""")
coverage.T.rename(columns={0: "value"})

,value
first_transaction,2010-01-01 00:01:00
last_transaction,2019-10-31 23:59:00
calendar_months,118
customers_transacting,1219
cards_used,4071
merchants,74831


In [9]:
edge_months = warehouse.query(con, """
    SELECT CAST(date_trunc('month', transaction_date) AS DATE) AS month_start,
           COUNT(DISTINCT CAST(transaction_date AS DATE))      AS days_with_activity,
           DAY(last_day(MIN(transaction_date)))                AS days_in_month
    FROM silver_transactions
    GROUP BY 1
    HAVING COUNT(DISTINCT CAST(transaction_date AS DATE)) < DAY(last_day(MIN(transaction_date)))
    ORDER BY 1
""")
print(f"{len(edge_months)} incomplete month(s) will be excluded from forecasting and variance analysis.")
edge_months

0 incomplete month(s) will be excluded from forecasting and variance analysis.


,month_start,days_with_activity,days_in_month


In [10]:
warehouse.query(con, """
    SELECT YEAR(transaction_date)                    AS year,
           COUNT(*)                                  AS transactions,
           SUM(amount)                               AS net_spend,
           AVG(CAST(is_refund AS INTEGER)) * 100     AS refund_pct,
           AVG(CAST(is_online AS INTEGER)) * 100     AS online_pct
    FROM silver_transactions
    GROUP BY 1 ORDER BY 1
""")

,year,transactions,net_spend,refund_pct,online_pct
0,2010,1240880,"54,232,556.12",5.16,10.87
1,2011,1290770,"55,778,904.96",5.08,10.81
2,2012,1321672,"56,832,410.86",5.01,11.15
3,2013,1352808,"58,284,939.62",4.98,11.62
4,2014,1365537,"58,617,820.51",4.92,11.82
5,2015,1388065,"59,514,007.43",4.90,12.33
6,2016,1392117,"59,844,028.90",4.91,12.31
7,2017,1399308,"59,628,480.63",4.93,12.15
8,2018,1394792,"59,627,317.94",4.84,12.17
9,2019,1159966,"49,475,055.31",4.91,12.16


## 3.5 Stage outcome

The silver layer holds five typed tables that follow the rules above, and the data-quality results are saved for reporting (stage 5). Any failed control listed in 3.3 is a known issue to resolve, not something to ignore.

Next: [Stage 4: Data Modelling & Analytics](../04-data-modelling-and-analytics/04-data-modelling-and-analytics.ipynb)